In [1]:
!nvidia-smi

Thu Jun 18 21:08:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import yaml, os

data_dir = '/content/drive/MyDrive/mma/mma-fighter-pose-estimation-dataset'

for split in ['train', 'valid', 'test']:
    img = os.path.join(data_dir, split, 'images')
    lbl = os.path.join(data_dir, split, 'labels')
    print(split, '| images:', len(os.listdir(img)), '| labels:', len(os.listdir(lbl)))

train | images: 3635 | labels: 3635
valid | images: 980 | labels: 980
test | images: 491 | labels: 491


In [5]:
fixed = {
    'path': data_dir,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['fighter'],
    'kpt_shape': [17, 3],
    'flip_idx': [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15],
}
fixed_path = '/content/data_fixed.yaml'
with open(fixed_path, 'w') as f:
    yaml.safe_dump(fixed, f, sort_keys=False)
print(yaml.safe_load(open(fixed_path)))

{'path': '/content/drive/MyDrive/mma/mma-fighter-pose-estimation-dataset', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 1, 'names': ['fighter'], 'kpt_shape': [17, 3], 'flip_idx': [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]}


In [7]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.4 MB/s eta 0:00:00


In [9]:
import shutil, os

SRC = '/content/drive/MyDrive/mma/mma-fighter-pose-estimation-dataset'
DST = '/content/mma-dataset'

shutil.copytree(SRC, DST)
print('copied to', DST)

for split in ['train', 'valid', 'test']:
    img = os.path.join(DST, split, 'images')
    lbl = os.path.join(DST, split, 'labels')
    print(split, '| images:', len(os.listdir(img)), '| labels:', len(os.listdir(lbl)))

copied to /content/mma-dataset
train | images: 3635 | labels: 3635
valid | images: 980 | labels: 980
test | images: 491 | labels: 491


In [10]:
import yaml
fixed = {
    'path': DST,                 # local disk now, not Drive
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['fighter'],
    'kpt_shape': [17, 3],
    'flip_idx': [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15],
}
fixed_path = '/content/data_fixed.yaml'
with open(fixed_path, 'w') as f:
    yaml.safe_dump(fixed, f, sort_keys=False)
print(yaml.safe_load(open(fixed_path)))

{'path': '/content/mma-dataset', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 1, 'names': ['fighter'], 'kpt_shape': [17, 3], 'flip_idx': [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]}


In [11]:
from ultralytics import YOLO
model = YOLO('yolo11n-pose.pt')
results = model.train(
    data=fixed_path,
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    patience=15,
    project='/content/drive/MyDrive/mma/runs',   # keep on Drive
    name='baseline',
)

Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15

In [12]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/mma/runs/baseline-2/weights/best.pt')

# run on the test split images
results = model.predict(
    source='/content/mma-dataset/test/images',
    save=True,
    conf=0.5,
    project='/content/drive/MyDrive/mma/runs',
    name='predict-test',
)
print('annotated images saved to Drive')


image 1/491 /content/mma-dataset/test/images/-vs-1-DPjtZWsn8gM-_mp4-0053_jpg.rf.37a513ffabe6b23d1de4508262aab281.jpg: 640x640 2 fighters, 39.0ms
image 2/491 /content/mma-dataset/test/images/-vs-1-DPjtZWsn8gM-_mp4-0085_jpg.rf.3a7f8d0889f4b1789a9b161ec56e9c7f.jpg: 640x640 2 fighters, 86.0ms
image 3/491 /content/mma-dataset/test/images/-vs-1-DPjtZWsn8gM-_mp4-0087_jpg.rf.63edfbd73fbbd91ecba53178543621fe.jpg: 640x640 2 fighters, 24.8ms
image 4/491 /content/mma-dataset/test/images/-vs-1-DPjtZWsn8gM-_mp4-0093_jpg.rf.624dfff5189708da214501430b643f3c.jpg: 640x640 2 fighters, 57.1ms
image 5/491 /content/mma-dataset/test/images/-vs-1-DPjtZWsn8gM-_mp4-0096_jpg.rf.eadf6f42cbf16a40a2c31d2822a28741.jpg: 640x640 2 fighters, 82.2ms
image 6/491 /content/mma-dataset/test/images/-vs-1-DPjtZWsn8gM-_mp4-0120_jpg.rf.0acc04493d03d8c21b4f38fb2c7c5bfa.jpg: 640x640 2 fighters, 83.9ms
image 7/491 /content/mma-dataset/test/images/-vs-1-DPjtZWsn8gM-_mp4-0125_jpg.rf.88118e8c271ca8b3fc8daf2eae70f6f7.jpg: 640x640 2 f